# Переранжирование

## Подход

**CatBoost с функцией потерь `YetiRank`**. 
Задача: для каждого запроса корректно упорядочить его ~500 кандидатов так, чтобы немногие релевантные оказались в начале.
`YetiRank` оптимизирует её напрямую, в отличие от LogLoss классификатора, который не знает, что кандидаты одного запроса конкурируют друг с другом за место в топ-50.

**Признаки** — переиспользуем почти всё, что уже посчитано
(`src/reranking_features.py`, `FEATURE_COLUMNS`):

- 5 сырых скоров сигналов кандидатогенерации (`params_coverage_score`,
  `text_coverage_score`, `text_similarity_score`, `location_match_score`,
  `microcat_match_score`) плюс сырое расстояние `location_distance_km` и
  сами `rrf_score`/`rrf_rank`;
- `item_rating`, `item_rating_reviews_count_log1p`, `item_price_log1p` (логарифм — оба сильно скошены), `item_is_phone_hidden`, `item_is_message_forbidden` — признаки из `feature_analysis.ipynb`, где для части из них уже показана связь с тем, что пользователи выбирают;
- `search_is_delivery_search` — флаг запроса, бесплатный, хотя и редкий.

**Обучающая выборка** — тот же трюк, что и для локальной валидации в `candidate_generation.ipynb`: прогоняем кандидатогенерацию на запросах из `train.parquet`, помечаем выбранное объявление `label=1`, остальные кандидаты того же запроса — `label=0`, группируем по `query_id`.

## 0. Импорты и конфигурация

In [5]:
import sys
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, ".")

from src.candidate_generator import CandidateGenerator
from src.config import CandidateGeneratorConfig
from src.data_loading import (
    build_local_eval_corpus,
    build_relevance_labels,
    load_dataset,
    sample_validation_queries,
)
from src.embeddings import E5Embedder
from src.evaluation import recall_at_k, relevant_item_coverage
from src.feature_loading import load_item_features, load_train_item_feature_lookup
from src.pipeline import build_indices
from src.reranker import rerank, to_pool, train_reranker
from src.reranking_features import FEATURE_COLUMNS, build_feature_table, prepare_item_content_features
from src.submission import build_answer, validate_answer
from src.text_processing import unique_tokens

pd.set_option("display.max_colwidth", 120)

DATA_DIR = "dataset"
MODEL_DIR = "models/multilingual-e5-large"
CACHE_DIR = "cache/embeddings"
PARAMS_TRUNCATE_CHARS = 200
DESCRIPTION_TRUNCATE_CHARS = 200

config = CandidateGeneratorConfig(top_k=500)
config

CandidateGeneratorConfig(top_k=500, unspecified_category_value=0, min_token_len=2, params_similarity_threshold=0.82, rrf_k=60, signal_weights={'params_coverage': 1.0, 'text_coverage': 1.0, 'text_similarity': 1.0, 'location_match': 1.0, 'microcat_match': 1.0}, location_decay_km=5.0, embedding_batch_size=256, random_seed=42)

## 1. Данные

In [6]:
data = load_dataset(DATA_DIR)
train = data["train"]
benchmark_items = data["benchmark_items"]
benchmark_queries = data["benchmark_queries"]

labels = build_relevance_labels(train)
print(f"Уникальных запросов в train: {len(labels):,}")

Уникальных запросов в train: 107,978


In [ ]:
N_VAL_QUERIES = 800

val_queries = sample_validation_queries(labels, n=N_VAL_QUERIES, seed=config.random_seed, max_relevant=200)

remaining_labels = labels[~labels["query_id"].isin(val_queries["query_id"])]

benchmark_item_ids = set(benchmark_items["item_id"])
reachable_mask = remaining_labels["relevant_item_ids"].map(
    lambda ids: len(ids & benchmark_item_ids) > 0
)
reachable_labels = remaining_labels[reachable_mask]
print(f"Обучающих запросов с релевантным объявлением в benchmark_items: "
      f"{len(reachable_labels):,} из {len(remaining_labels):,}")

N_TRAIN_QUERIES = len(reachable_labels)
train_queries = sample_validation_queries(reachable_labels, n=N_TRAIN_QUERIES, seed=777, max_relevant=200)

assert set(val_queries["query_id"]).isdisjoint(train_queries["query_id"])
print(f"Обучающих запросов: {len(train_queries):,}, валидационных: {len(val_queries):,}")

Обучающих запросов с релевантным объявлением в benchmark_items: 15,425 из 107,178
Обучающих запросов: 15,217, валидационных: 800


## 2. Корпус и признаки объявлений

Локальный корпус — `benchmark_items` плюс недостающие истинные объявления **валидационных** запросов.

In [ ]:
eval_corpus = build_local_eval_corpus(benchmark_items, train, val_queries)
print(f"Локальный корпус: {len(eval_corpus):,} объявлений "
      f"(добавлено сверх benchmark_items: {len(eval_corpus) - len(benchmark_items):,})")

relevance = dict(zip(labels["query_id"], labels["relevant_item_ids"]))

val_relevance = {qid: relevance[qid] for qid in val_queries["query_id"]}
val_coverage_check = relevant_item_coverage(val_relevance, eval_corpus["item_id"])
print(f"Валидационные запросы — доля релевантных объявлений в корпусе: {val_coverage_check:.4%}")
assert val_coverage_check == 1.0, "у val_queries все релевантные объявления должны быть в корпусе"

train_relevance = {qid: relevance[qid] for qid in train_queries["query_id"]}
train_coverage_check = relevant_item_coverage(train_relevance, eval_corpus["item_id"])
print(f"Обучающие запросы — доля релевантных объявлений в корпусе: {train_coverage_check:.4%}")

Локальный корпус: 191,311 объявлений (добавлено сверх benchmark_items: 2,099)
Валидационные запросы — доля релевантных объявлений в корпусе: 100.0000%
Обучающие запросы — доля релевантных объявлений в корпусе: 13.3983%


Признаки объявлений (рейтинг, отзывы, цена, флаги) — из
`feature_loading`: для объявлений из `benchmark_items` напрямую, для
добавленных "из train" — через отдельный поиск по `train.parquet`
(`load_train_item_feature_lookup`), т.к. `benchmark_items.parquet` их не
содержит.

In [9]:
item_content_raw = pd.concat(
    [load_item_features(DATA_DIR), load_train_item_feature_lookup(DATA_DIR)],
    ignore_index=True,
).drop_duplicates(subset="item_id", keep="first")

item_content_features = prepare_item_content_features(item_content_raw)
print(f"Признаков объявлений подготовлено: {len(item_content_features):,}")
item_content_features.head(3)

Признаков объявлений подготовлено: 515,895


,item_id,item_rating,item_rating_reviews_count_log1p,item_price_log1p,item_is_phone_hidden,item_is_message_forbidden
0,111eb8b979577d79,5.0,4.060443,6.216606,0,0
1,75fc8e10f5a66fc4,5.0,2.079442,6.803505,0,0
2,3f6ae81704565b9c,5.0,2.708050,6.908755,1,0


## 3. Эмбеддинги и индексы

Для `benchmark_items` эмбеддинги уже посчитаны и закешированы
`candidate_generation.ipynb` (те же ключи кеша — переиспользуются мгновенно).
Отдельно кодируем только добавленные объявления.

In [10]:
embedder = E5Embedder(MODEL_DIR, cache_dir=CACHE_DIR, batch_size=128, show_progress=True)

t0 = time.time()
benchmark_dense_texts = (
    benchmark_items["item_title_raw"].fillna("")
    + " "
    + benchmark_items["item_description_raw"].fillna("").str.slice(0, DESCRIPTION_TRUNCATE_CHARS)
).tolist()
benchmark_dense_emb = embedder.encode(benchmark_dense_texts, prefix="passage", cache_key="benchmark_item_title_description")
benchmark_params_emb = embedder.encode(
    benchmark_items["item_infm_params_text"].fillna("").str.slice(0, PARAMS_TRUNCATE_CHARS).tolist(),
    prefix="passage",
    cache_key="benchmark_item_params",
)
print(f"benchmark_items: {time.time()-t0:.1f}s, {benchmark_dense_emb.shape}")

benchmark_items: 1.0s, (189212, 1024)


In [11]:
n_benchmark_items = len(benchmark_items)
extra_items = eval_corpus.iloc[n_benchmark_items:]
print(f"Дополнительных объявлений: {len(extra_items):,}")

if len(extra_items):
    extra_dense_texts = (
        extra_items["item_title_raw"].fillna("")
        + " "
        + extra_items["item_description_raw"].fillna("").str.slice(0, DESCRIPTION_TRUNCATE_CHARS)
    ).tolist()
    extra_dense_emb = embedder.encode(extra_dense_texts, prefix="passage", cache_key="rerank_extra_item_title_description")
    extra_params_emb = embedder.encode(
        extra_items["item_infm_params_text"].fillna("").str.slice(0, PARAMS_TRUNCATE_CHARS).tolist(),
        prefix="passage",
        cache_key="rerank_extra_item_params",
    )
    eval_dense_emb = np.concatenate([benchmark_dense_emb, extra_dense_emb])
    eval_params_emb = np.concatenate([benchmark_params_emb, extra_params_emb])
else:
    eval_dense_emb = benchmark_dense_emb
    eval_params_emb = benchmark_params_emb

assert eval_dense_emb.shape[0] == len(eval_corpus)
assert eval_params_emb.shape[0] == len(eval_corpus)

Дополнительных объявлений: 2,099


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

In [12]:
rerank_query_vocab = sorted(
    unique_tokens(train_queries["search_infm_params_text"])
    | unique_tokens(val_queries["search_infm_params_text"])
    | unique_tokens(benchmark_queries["search_infm_params_text"])
)
print(f"Словарь токенов параметров запросов: {len(rerank_query_vocab)}")

eval_params_index, eval_text_index, eval_dense_index, eval_location_index, eval_microcat_index = build_indices(
    eval_corpus, eval_params_emb, eval_dense_emb, rerank_query_vocab, embedder, config,
    query_cache_key="rerank_query_vocab_params",
)
eval_generator = CandidateGenerator(
    eval_corpus, eval_params_index, eval_text_index, eval_dense_index, eval_location_index, eval_microcat_index, config
)
print("Индексы готовы.")

Словарь токенов параметров запросов: 547


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Индексы готовы.


## 4. Признаки и метки для обучения

Для каждого запроса — до 500 кандидатов с сырыми скорами сигналов.
Метка `1` у объявления, которое пользователь реально выбрал для этого запроса, `0` — у остальных кандидатов того же пула.

In [13]:
train_query_embeddings = embedder.encode(
    train_queries["search_query"].fillna("").tolist(), prefix="query", cache_key="rerank_train_query_text_emb"
)
val_query_embeddings = embedder.encode(
    val_queries["search_query"].fillna("").tolist(), prefix="query", cache_key="val_query_text_emb"
)

Batches:   0%|          | 0/119 [00:00<?, ?it/s]

In [14]:
t0 = time.time()
train_candidates = eval_generator.generate_with_features(train_queries, train_query_embeddings)
print(f"Кандидаты (RRF) для {len(train_queries)} обучающих запросов: {time.time()-t0:.1f}s, {len(train_candidates):,} строк")

Generating candidates:   0%|          | 0/15217 [00:00<?, ?it/s]

Кандидаты (RRF) для 15217 обучающих запросов: 1061.6s, 7,608,500 строк


In [15]:
t0 = time.time()
val_candidates = eval_generator.generate_with_features(val_queries, val_query_embeddings)
print(f"Кандидаты (RRF) для {len(val_queries)} валидационных запросов: {time.time()-t0:.1f}s, {len(val_candidates):,} строк")

Generating candidates:   0%|          | 0/800 [00:00<?, ?it/s]

Кандидаты (RRF) для 800 валидационных запросов: 55.7s, 400,000 строк


In [16]:
QUERY_FEATURE_COLUMNS = ["query_id", "search_is_delivery_search"]

train_table = build_feature_table(
    train_candidates, item_content_features, relevance, query_features=train_queries[QUERY_FEATURE_COLUMNS]
)
val_table = build_feature_table(
    val_candidates, item_content_features, relevance, query_features=val_queries[QUERY_FEATURE_COLUMNS]
)

print(f"train_table: {len(train_table):,} строк, положительных меток: {train_table['label'].sum():,}")
print(f"val_table: {len(val_table):,} строк, положительных меток: {val_table['label'].sum():,}")
val_relevance = {qid: relevance[qid] for qid in val_queries["query_id"]}
train_table[["query_id", "item_id"] + list(FEATURE_COLUMNS) + ["label"]].head(3)

train_table: 7,608,500 строк, положительных меток: 20,375
val_table: 400,000 строк, положительных меток: 1,909


,query_id,item_id,params_coverage_score,text_coverage_score,text_similarity_score,location_match_score,location_distance_km,microcat_match_score,rrf_score,rrf_rank,item_rating,item_rating_reviews_count_log1p,item_price_log1p,item_is_phone_hidden,item_is_message_forbidden,search_is_delivery_search,label
0,train_000007,a9cb6117336842d1,0.0,0.0,0.842888,0.0,NaN,0.804059,0.016400,1,4.762063,6.400257,6.196444,0,0,0,0
1,train_000007,8d40860aabafcbb8,0.0,0.0,0.828835,0.0,NaN,0.821503,0.016180,2,5.000000,3.367296,6.398595,0,0,0,0
2,train_000007,b3f1230c3a107e68,0.0,0.0,0.828508,0.0,NaN,0.815118,0.015889,3,NaN,0.693147,6.908755,1,0,0,0


## 5. Обучение CatBoost

In [ ]:
train_pool = to_pool(train_table, FEATURE_COLUMNS, label_column="label")
val_pool = to_pool(val_table, FEATURE_COLUMNS, label_column="label")

t0 = time.time()
model = train_reranker(train_pool, val_pool, iterations=300, learning_rate=0.05, depth=6)
print(f"Обучение заняло {time.time()-t0:.1f}s, лучшая итерация: {model.get_best_iteration()}")

0:	learn: 0.6200911	test: 0.7087908	best: 0.7087908 (0)	total: 2.01s	remaining: 10m
100:	learn: 0.7434668	test: 0.8335879	best: 0.8335879 (99)	total: 2m 48s	remaining: 5m 31s
200:	learn: 0.7491622	test: 0.8334397	best: 0.8360885 (148)	total: 5m 29s	remaining: 2m 42s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8360884996
bestIteration = 148

Shrink model to first 149 iterations.
Обучение заняло 406.8s, лучшая итерация: 148


## 6. Recall@50

Бейзлайн — просто верхние 50 по исходному RRF. 
Реранкер — предсказанный CatBoost скор, топ-50 по нему на том же пуле из 500 кандидатов.

In [18]:
baseline_predictions = {
    query_id: group.loc[group["rrf_rank"] <= 50, "item_id"].tolist()
    for query_id, group in val_table.groupby("query_id", sort=False)
}
reranked_predictions = rerank(model, val_table, FEATURE_COLUMNS, top_k=50)

baseline_summary, _ = recall_at_k(baseline_predictions, val_relevance, ks=(50,))
reranked_summary, _ = recall_at_k(reranked_predictions, val_relevance, ks=(50,))

comparison = pd.DataFrame(
    {
        "variant": ["RRF top-50 (бейзлайн)", "CatBoost YetiRank top-50"],
        "recall@50": [baseline_summary["recall_at_k"].iloc[0], reranked_summary["recall_at_k"].iloc[0]],
    }
)
comparison

,variant,recall@50
0,RRF top-50 (бейзлайн),0.511462
1,CatBoost YetiRank top-50,0.744462


## 7. Важность признаков

In [19]:
importances = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "importance": model.get_feature_importance(train_pool),
    }
).sort_values("importance", ascending=False).reset_index(drop=True)
importances

,feature,importance
0,location_match_score,0.097225
1,text_coverage_score,0.039202
2,text_similarity_score,0.029259
3,params_coverage_score,0.017361
4,rrf_score,0.016157
5,location_distance_km,0.013546
6,item_rating_reviews_count_log1p,0.010980
7,rrf_rank,0.003324
8,item_rating,0.002882
9,item_price_log1p,0.001309


## 8. Применение к тесту

In [21]:
benchmark_params_index, benchmark_text_index, benchmark_dense_index, benchmark_location_index, benchmark_microcat_index = build_indices(
    benchmark_items, benchmark_params_emb, benchmark_dense_emb, rerank_query_vocab, embedder, config,
    query_cache_key="rerank_query_vocab_params",
)
benchmark_generator = CandidateGenerator(
    benchmark_items, benchmark_params_index, benchmark_text_index, benchmark_dense_index,
    benchmark_location_index, benchmark_microcat_index, config,
)

benchmark_query_embeddings = embedder.encode(
    benchmark_queries["search_query"].fillna("").tolist(), prefix="query", cache_key="benchmark_query_text_emb"
)

t0 = time.time()
benchmark_candidates = benchmark_generator.generate_with_features(benchmark_queries, benchmark_query_embeddings)
print(f"Кандидаты для {len(benchmark_queries)} запросов бенчмарка: {time.time()-t0:.1f}s")

benchmark_table = build_feature_table(
    benchmark_candidates, item_content_features, relevance=None,
    query_features=benchmark_queries[QUERY_FEATURE_COLUMNS],
)
benchmark_predictions = rerank(model, benchmark_table, FEATURE_COLUMNS, top_k=50)
print(f"Переранжировано запросов: {len(benchmark_predictions)}")

Generating candidates:   0%|          | 0/2452 [00:00<?, ?it/s]

Кандидаты для 2452 запросов бенчмарка: 162.9s
Переранжировано запросов: 2452


In [23]:
answer_df = build_answer(benchmark_predictions, benchmark_queries["query_id"].tolist(), max_candidates=50)

problems = validate_answer(answer_df, benchmark_queries["query_id"].tolist(), benchmark_items["item_id"].tolist())
if problems:
    print("НАЙДЕНЫ ПРОБЛЕМЫ:")
    for problem in problems:
        print(" -", problem)
else:
    print("Формат answer.csv в порядке.")

answer_df.to_csv("answer.csv", index=False)
print(f"Сохранено: answer.csv ({len(answer_df)} строк)")
answer_df.head(3)

Формат answer.csv в порядке.
Сохранено: answer.csv (2452 строк)


,query_id,answer
0,70DfDUpwjxB4lzFd,6047e1566a959d3a 9515d1e1ecdac72c 355392014208b7bf 14d535285758af89 bbfd9a8c71d3bd2b d01e14d2ffd4ddba f4e8903d7689e2...
1,JTrdTaZJvSiLPkXj,dfa239d031b11832 7e18adf210484046 1cc96086d662cffc 2f09017b0819844e 0ecca189a4823554 8f74ccd5549146c6 0529307cdfbe4a...
2,LZCZNoVG4AFUkVRJ,d8fce513e4f000a7 10eb6d6da3230bbb edecb3695ab88901 d62074dd39caefee 168a9207e80b0be4 dab52187b4500d9b c1abf382ab9a79...
